# 07 — All models through the search knobs

Goal: run every registered model through the model-side knobs of
`run_experiment` on the named feature set `feature_set="selected"`:
`search="grid"` / random search, alternative `scoring`, `repeats=1` +
`save=False` for fast checks, the `estimator=` escape hatch, and
`fit_full=True` for a submission. Thin driver only — the definitions live in
`src/daa/models/`, `src/daa/experiment.py`, and `src/daa/evaluation/` (see
`docs/models-guide.md`). Restart & Run All, then strip outputs before
committing (the `nbstripout` hook enforces it).


In [ ]:
import json

import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

from daa.config import OUTPUTS_DIR, RUNS_DIR
from daa.data import load_test, load_train
from daa.evaluation import load_run, validate_submission, write_submission_from_result
from daa.experiment import run_experiment
from daa.models.registry import MODEL_NAMES
from daa.models.spaces import SEARCH_SPACES
from daa.splits import get_folds


In [ ]:
X, y = load_train()
folds = get_folds(y)
print(f"real data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")
print(f"{len(folds)} folds, {y.nunique()} classes")
print(f"models: {', '.join(MODEL_NAMES)}")
print(f"with search spaces: {', '.join(SEARCH_SPACES)}")


## The knobs this notebook exercises

All of them are parameters of `daa.experiment.run_experiment`:

- `feature_set="selected"` — resolves the shortlist from
  `outputs/selected_features.json` by name (`"all"` is the default); the
  column list stays with the feature-selection owner.
- `search=` — `"grid"`, `"random"`, or a dict
  `{"kind", "space", "n_iter", "scoring", "n_jobs"}`. It wraps the pipeline
  in `GridSearchCV` / `RandomizedSearchCV` with `refit=True`; `run_cv` clones
  per fold, so the search runs as honest nested CV. The string forms use
  `SEARCH_SPACES[model_name]`.
- `repeats=` — keep only the first N repeats of the frozen folds. Fast
  iteration only; combine with `save=False` so a partial run never lands on
  the leaderboard as if it were the full 3x5 scheme.
- `estimator=` — a pre-built, unfitted estimator used directly as the `clf`
  step (calibrated / wrapped / ensemble). With `search`, pass a dict with an
  explicit `space`, because there is no `model_name` to look one up.
- `fit_full=True` — also fit on all rows and store the estimator in the run
  dir; required for a submission.
- `preprocessor=None` (the default) — the encoding owner's
  `features.preprocess.DEFAULT_VARIANT`; not named here on purpose.


## Fast check first: `repeats=1, save=False`

One grid-search pass over a single repeat of the folds, not saved. This is
the call to use while iterating: cheap, and nothing partial reaches the
leaderboard.


In [ ]:
smoke = run_experiment(
    "decision_tree",
    {"max_depth": 16},
    name="07_smoke_tree",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    search="grid",
    repeats=1,
    save=False,
)
pd.Series(smoke.metrics, name=smoke.name)


## Grid search on every tunable model

Every model with a registered `SEARCH_SPACES` entry gets `search="grid"` on
the full frozen folds and the named feature set. (`dummy` has no space — it
has nothing to tune.) Each run is saved, so it appends a leaderboard row with
the search config recorded in `params["search"]`.


In [ ]:
grid_results = {}
for model_name in SEARCH_SPACES:
    result = run_experiment(
        model_name,
        name=f"07_{model_name}_selected_grid",
        X=X,
        y=y,
        folds=folds,
        feature_set="selected",
        search="grid",
    )
    grid_results[model_name] = result
    print(f"{model_name:>12}  macro_f1={result.metrics['macro_f1_mean']:.4f}")

grid_summary = pd.DataFrame(
    [{"model": name, **result.metrics} for name, result in grid_results.items()]
).sort_values("macro_f1_mean", ascending=False)
grid_summary


## Randomized search and alternative scoring

`{"kind": "random", "n_iter": 12}` samples 12 candidates from the registered
space instead of evaluating all of them (deterministic via `SEED`).
`scoring=` picks the refit criterion from `daa.evaluation.metrics.SCORERS`;
the default `"macro_f1"` is the competition metric, `"qwk"` is the
ordinal-aware alternative. Below: random search on `hist_gb`, and a `logreg`
grid refit by QWK.


In [ ]:
random_result = run_experiment(
    "hist_gb",
    name="07_hist_gb_selected_random",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    search={"kind": "random", "n_iter": 12},
)
qwk_result = run_experiment(
    "logreg",
    name="07_logreg_selected_grid_qwk",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    search={"kind": "grid", "scoring": "qwk"},
)
pd.DataFrame([{"run": r.name, **r.metrics} for r in (random_result, qwk_result)])


## The `estimator=` escape hatch

`run_experiment` normally builds the `clf` step from the registry via
`model_name`. Pass `estimator=` to supply an already-constructed, unfitted
estimator instead — here a probability-calibrated logistic regression. The
recorded `params["model"]` is then the class name, and `search` must carry an
explicit `space` (still `clf__`-prefixed; here it reaches through the
calibration wrapper to the inner estimator's `C`).


In [ ]:
calibrated = run_experiment(
    name="07_calibrated_logreg_selected",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    estimator=CalibratedClassifierCV(LogisticRegression(max_iter=2000)),
    search={"kind": "grid", "space": {"clf__estimator__C": [0.1, 1.0, 10.0]}},
)
print("recorded model:", calibrated.params["model"])
pd.Series(calibrated.metrics, name=calibrated.name)


## Full fit and a submission from a search run

`fit_full=True` fits the search pipeline on all rows and stores it as
`estimator.joblib` in the run dir. Because `build_search` always sets
`refit=True`, the stored search object predicts directly, so
`write_submission_from_result` works unchanged. Below: re-run the best grid
model with `fit_full=True`, write and validate a submission, then reload the
run with `load_run` and confirm the artifacts round-trip.


In [ ]:
best_model = grid_summary.iloc[0]["model"]
final = run_experiment(
    best_model,
    name=f"07_{best_model}_selected_grid_full",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    search="grid",
    fit_full=True,
)
X_test, row_ids = load_test()
submission_path = write_submission_from_result(
    final, X_test, row_ids, OUTPUTS_DIR / "submission_07_search.csv"
)
validate_submission(submission_path, row_ids)
print(f"wrote and validated {submission_path}")

reloaded = load_run(RUNS_DIR / final.run_id)
reloaded.metrics == final.metrics and reloaded.params == final.params


## Leaderboard: everything this notebook ran

Rows are identifiable by the `07_` name prefix. `params` carries `model`,
`feature_set`, `search`, and `repeats` (when used), so comparable rows are
recognizable at a glance.


In [ ]:
leaderboard = pd.read_csv(OUTPUTS_DIR / "leaderboard.csv")
ours = leaderboard[leaderboard["name"].str.startswith("07_")]

records = []
for _, row in ours.iterrows():
    params = json.loads(row["params"])
    search = params.get("search") or {}
    records.append(
        {
            "run_id": row["run_id"],
            "name": row["name"],
            "model": params.get("model"),
            "feature_set": params.get("feature_set"),
            "search": search.get("kind", "-"),
            "repeats": params.get("repeats", "-"),
            "n_features": row["n_features"],
            "macro_f1_mean": round(row["macro_f1_mean"], 4),
            "qwk_mean": round(row["qwk_mean"], 4),
        }
    )
pd.DataFrame(records)


## Next

- Widen or prune the grids in `daa/models/spaces.py` (the models track owns
  them) and re-run — same folds, same feature set, so rows stay comparable.
- Compare against `feature_set="all"` rows: the `feature_hash` in `params`
  tells you which columns a row used.
- Error analysis on the best run:
  `load_run(RUNS_DIR / final.run_id).oof` has the out-of-fold predictions,
  `per_class` and `confusion` the breakdowns.
